# Product Dataset Cleaning & Exploration

In [1]:
import pandas as pd
import numpy as np

## 1. Loading the Data

In [2]:
df = pd.read_csv('Combined_dataset.csv')
print(f"Raw dataset shape: {df.shape}")
df.head(3)

Raw dataset shape: (1000, 24)


,product_id,title,product_description,rating,ratings_count,initial_price,discount,final_price,currency,images,...,amount_of_stars,what_customers_said,seller_name,sizes,videos,seller_information,variations,best_offer,more_offers,category
0,8376765,Lino Perros,Women Navy Blue Solid Backpack,3.8,15,3995,58.0,"""₹3,995.00""",INR,http://assets.myntassets.com/assets/images/837...,...,"{""1_star"":2,""2_stars"":0,""3_stars"":3,""4_stars"":...",NaN,NaN,"[{""size"":""Onesize""}]","[""rw-8376765_cae700""]",NaN,[{}],{},"[{""offer_name"":""10% Instant Discount on Citi C...",backpacks
1,9136281,Tommy Hilfiger,Unisex Navy Blue Striped Backpack,4.5,67,2899,35.0,"""₹2,899.00""",INR,http://assets.myntassets.com/assets/images/913...,...,"{""1_star"":3,""2_stars"":4,""3_stars"":2,""4_stars"":...",NaN,NaN,"[{""size"":""Onesize""}]","[""rw-9136281_cae700""]",NaN,"[{},{}]",{},"[{""offer_name"":""10% Instant Discount on Citi C...",backpacks
2,17633752,Lavie,Aries Women Pink Mini Backpack,4.4,226,2999,65.0,"""₹2,999.00""",INR,http://assets.myntassets.com/assets/images/176...,...,"{""1_star"":9,""2_stars"":5,""3_stars"":10,""4_stars""...",NaN,NaN,"[{""size"":""S""}]","[""https://videos.myntassets.com/assets/videos/...",NaN,"[{},{},{},{},{},{}]",{},"[{""offer_name"":""10% Instant Discount on Citi C...",backpacks


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   product_id              1000 non-null   int64  
 1   title                   1000 non-null   str    
 2   product_description     1000 non-null   str    
 3   rating                  1000 non-null   float64
 4   ratings_count           1000 non-null   int64  
 5   initial_price           1000 non-null   int64  
 6   discount                879 non-null    float64
 7   final_price             1000 non-null   str    
 8   currency                1000 non-null   str    
 9   images                  1000 non-null   str    
 10  delivery_options        1000 non-null   str    
 11  product_details         1000 non-null   str    
 12  breadcrumbs             1000 non-null   str    
 13  product_specifications  1000 non-null   str    
 14  amount_of_stars         1000 non-null   str    
 15 

## 2.Missing Values

In [4]:
null_counts = df.isnull().sum()
null_counts[null_counts > 0]

discount               121
what_customers_said    573
seller_name            301
videos                 781
seller_information     301
variations             562
dtype: int64

In [5]:
# Fill missing ratings and counts
df['rating'] = df['rating'].fillna(df['rating'].median())
df['ratings_count'] = df['ratings_count'].fillna(0)

# Drop rows with missing critical info
df = df.dropna(subset=['title', 'initial_price'])
print(f"Dataset shape after handling nulls: {df.shape}")

Dataset shape after handling nulls: (1000, 24)


## 3. Filtering and Column Selection

In [6]:
# Filter highly-rated premium items
premium_df = df[(df['rating'] >= 4.0) & (df['initial_price'] > 1000)].copy()

# Keep only relevant columns
cols_to_keep = ['product_id', 'title', 'rating', 'ratings_count', 'initial_price', 'category']
premium_df = premium_df[cols_to_keep]
print(f"Shape after filtering: {premium_df.shape}")
premium_df.head()

Shape after filtering: (505, 6)


,product_id,title,rating,ratings_count,initial_price,category
1,9136281,Tommy Hilfiger,4.5,67,2899,backpacks
2,17633752,Lavie,4.4,226,2999,backpacks
3,1376949,F Gear,4.4,1052,1675,backpacks
4,13939916,MYTRIDENT,4.7,12,2899,bath-robe
5,17198778,H&M,4.5,42,1399,bathroom-accessories


## 4. Removing Duplicates

In [7]:
# Check duplicates count
duplicates = premium_df.duplicated(subset=['product_id']).sum()
print(f"Duplicate product IDs found: {duplicates}")

# Remove duplicates
cleaned_df = premium_df.drop_duplicates(subset=['product_id'], keep='first').copy()
print(f"Shape after dropping duplicates: {cleaned_df.shape}")

Duplicate product IDs found: 0
Shape after dropping duplicates: (505, 6)


## 5.Derived Columns

In [8]:
np.random.seed(42)

# Set price
cleaned_df['price'] = cleaned_df['initial_price']
cleaned_df['quantity'] = np.random.randint(1, 11, size=len(cleaned_df))

# Calculate total amount
cleaned_df['total_amount'] = cleaned_df['price'] * cleaned_df['quantity']

cleaned_df[['product_id', 'title', 'price', 'quantity', 'total_amount']].head()

,product_id,title,price,quantity,total_amount
1,9136281,Tommy Hilfiger,2899,7,20293
2,17633752,Lavie,2999,4,11996
3,1376949,F Gear,1675,8,13400
4,13939916,MYTRIDENT,2899,5,14495
5,17198778,H&M,1399,7,9793


## 6. Exporting the Cleaned Data

In [9]:
cleaned_df.to_csv('cleaned_dataset.csv', index=False)
print("Cleaned dataset successfully exported to 'cleaned_dataset.csv'")

Cleaned dataset successfully exported to 'cleaned_dataset.csv'
